In [1]:
import os
import sys
import math
import pathlib
import json
import pickle
import numpy as np
import matplotlib.pyplot as plt


os.environ['MS_ENABLE_TRE'] = '0'
from tqdm import tqdm

import mindspore as ms
from mindspore import  ops
from mindquantum import  Simulator
from mindquantum.core.gates import RY
from mindquantum.core.circuit import Circuit
from mindspore import nn
from mindquantum.framework import MQLayer,MQOps        
                      
project_path = pathlib.Path.cwd().parent.parent
sys.path.append(f"{project_path}/src")
print(f"{project_path}/src")

from datasets_utils import get_tree_dataloaders,get_dataloaders
from model import QuantumCircuit
from train_utils import Trainer
from adaboost import QAdaBoost


/home/qcql/code/QAdaboost_Huawei/src


In [2]:
# Args
class Args:
    config = f"{project_path}/configs/tree_classification_012.json"  # 修改为实际的config文件路径
    seed = 0  # 修改为实际的seed数值

args = Args()

seed = args.seed
with open(args.config, 'r') as f:
    config = json.load(f)

In [3]:
label_binary_tree = pickle.load(open(config['label_binary_tree_path'], 'rb'))
class_idx_list_0 = config["class_idx_list_0"]
class_idx_list_1 = config["class_idx_list_1"]
data_type = config["dataset_config"]["data_type"]
n_qubits = config["model_config"]["n_qubits"]
n_layers = config["model_config"]["n_layers"]
n_train_samples = config["dataset_config"]["n_train_samples"]
n_test_samples = config["dataset_config"]["n_test_samples"]
data_type = config["dataset_config"]["data_type"]
batch_size = config["dataset_config"]["batch_size"]
train_config = config["train_config"]
train_config["logging_config"]["wandb_config"]["tags"] = [f"seed_{seed}"]
train_config["logging_config"]["wandb_config"]["experiment_name"] = f"Adaboost_Classification_012_seed_{seed}"

In [4]:
ms.set_seed(seed)                                     # 设置生成随机数的种子
ms.set_context(mode=ms.PYNATIVE_MODE)
ms.set_device("CPU")  

## Training for 0/1 vs 2

In [5]:
train_loader_012,test_loader_012 = get_tree_dataloaders(n_qubits=n_qubits,n_layers=1,n_train_samples=n_train_samples,n_test_samples=n_test_samples,batch_size=batch_size,data_type=data_type,label_binary_tree=label_binary_tree,class_idx_list=class_idx_list_0)

In [6]:
n_train_012 = len(train_loader_012) * batch_size
adaboost_012 = QAdaBoost(n_train=n_train_012)

In [7]:

base_classifier_count = 0
while True:
    
    weights = adaboost_012.get_weights()
    
    quantum_circuit = QuantumCircuit(n_qubits=n_qubits,n_layers=n_layers)
    quantum_grad_ops = quantum_circuit.get_grad_ops()
    model = MQLayer(quantum_grad_ops)
    

    

    trainer = Trainer(config,model,weights,train_loader_012)
    model,train_error,predictions,labels = trainer.train()
    
    adaboost_012.record(model,train_error)
    adaboost_012.update_weights(labels,predictions)
    base_classifier_count += 1
    error_bound = adaboost_012.get_error_bound(tight=True)
    
    print(f"Train error: {train_error}")
    print(f"Error bound: {error_bound}")
    
    if train_error >= 0.5:
        print(f"Base classifier {base_classifier_count} has error greater than 0.5, unable to learn further, stopping training")
        break
    if train_error == 0.0:
        print(f"Base classifier {base_classifier_count} has error 0.0, learning completed, stopping training")
        break;
    if error_bound < 0.1: # just for example 
        print(f"Base classifier {base_classifier_count} has error bound less than 0.1 (just for example), requirement met, stopping training")
        break
    # adaboost.save(adaboost_path)
    

Training:   0%|          | 0/100 [00:00<?, ?it/s][WARNING] ME(1534277,78e859c50600,python):2026-01-27-22:48:05.958.009 [mindspore/ccsrc/tools/error_handler/error_config.cc:215] operator()] Value of `MS_ENABLE_TFT` is ``
[WARNING] ME(1534277,78e859c50600,python):2026-01-27-22:48:05.958.018 [mindspore/ccsrc/tools/error_handler/error_config.cc:163] operator()] Can find `TRE` in environment var `MS_ENABLE_TFT`
Training:  18%|█▊        | 18/100 [00:10<00:47,  1.73it/s]


Recorded estimator 1 with error 0.0632
sum of weights:  1.0000000123555381
Train error: 0.06316666502971202
Error bound: 0.6827362041827743


Training:  13%|█▎        | 13/100 [00:07<00:50,  1.72it/s]


Recorded estimator 2 with error 0.1184
sum of weights:  1.0000000211729452
Train error: 0.11839530151337385
Error bound: 0.510231356007437


Training:  26%|██▌       | 26/100 [00:14<00:42,  1.74it/s]


...

Recorded estimator 5 with error 0.0096
sum of weights:  0.9999998880518394
Train error: 0.009574260067893192
Error bound: 0.1560459415856156


Training:  17%|█▋        | 17/100 [00:09<00:47,  1.76it/s]


Recorded estimator 6 with error 0.3339
sum of weights:  0.9999999093207503
Train error: 0.333920884411782
Error bound: 0.14767086556766992


Training:  21%|██        | 21/100 [00:11<00:44,  1.77it/s]

Recorded estimator 7 with error 0.0503
sum of weights:  0.9999999773103869
Train error: 0.050251890264917165
Error bound: 0.09853769758381711
Base classifier 7 has error bound less than 0.1 (just for example), requirement met, stopping training


In [11]:
total_acc = 0
for batch in test_loader_012:
    data,labels = batch
    predictions = adaboost_012(data).squeeze()
    batch_acc = np.sum(predictions == labels) / len(labels)
    total_acc += batch_acc
test_acc = total_acc / len(test_loader_012)
print(f"Test accuracy: {test_acc}")


Test accuracy: 1.0


## Training for 0 vs 1

In [8]:
train_loader_01,test_loader_01 = get_tree_dataloaders(n_qubits=n_qubits,n_layers=1,n_train_samples=n_train_samples,n_test_samples=n_test_samples,batch_size=batch_size,data_type=data_type,label_binary_tree=label_binary_tree,class_idx_list=class_idx_list_1)

In [9]:
n_train_01 = len(train_loader_01) * batch_size
adaboost_01 = QAdaBoost(n_train=n_train_01)

In [10]:

base_classifier_count = 0
while True:
    
    weights = adaboost_01.get_weights()
    
    quantum_circuit = QuantumCircuit(n_qubits=n_qubits,n_layers=n_layers)
    quantum_grad_ops = quantum_circuit.get_grad_ops()
    model = MQLayer(quantum_grad_ops)
    

    

    trainer = Trainer(config,model,weights,train_loader_01)
    model,train_error,predictions,labels = trainer.train()
    
    adaboost_01.record(model,train_error)
    adaboost_01.update_weights(labels,predictions)
    base_classifier_count += 1
    error_bound = adaboost_01.get_error_bound(tight=True)
    
    print(f"Train error: {train_error}")
    print(f"Error bound: {error_bound}")
    
    if train_error >= 0.5:
        print(f"Base classifier {base_classifier_count} has error greater than 0.5, unable to learn further, stopping training")
        break
    if train_error == 0.0:
        print(f"Base classifier {base_classifier_count} has error 0.0, learning completed, stopping training")
        break;
    if error_bound < 0.1: # just for example 
        print(f"Base classifier {base_classifier_count} has error bound less than 0.1 (just for example), requirement met, stopping training")
        break
    # adaboost.save(adaboost_path)
    

Training:  25%|██▌       | 25/100 [00:09<00:28,  2.61it/s]


Recorded estimator 1 with error 0.1623
sum of weights:  1.0000000449533
Train error: 0.1622500019147992
Error bound: 0.7960047527110081


Training:  50%|█████     | 50/100 [00:18<00:18,  2.70it/s]


Recorded estimator 2 with error 0.1244
sum of weights:  1.0000000285272785
Train error: 0.12444046977907419
Error bound: 0.6003514508152348


Training:  83%|████████▎ | 83/100 [00:30<00:06,  2.72it/s]


...

Recorded estimator 12 with error 0.2826
sum of weights:  0.9999999276881341
Train error: 0.28264832496643066
Error bound: 0.11820562672041944


Training:  14%|█▍        | 14/100 [00:05<00:33,  2.56it/s]


Recorded estimator 13 with error 0.2475
sum of weights:  0.9999999288478404
Train error: 0.24745419435203075
Error bound: 0.1040495199649994


Training:  20%|██        | 20/100 [00:07<00:30,  2.61it/s]

Recorded estimator 14 with error 0.2902
sum of weights:  0.9999998982707466
Train error: 0.29023449681699276
Error bound: 0.09528418113321328
Base classifier 14 has error bound less than 0.1 (just for example), requirement met, stopping training


In [11]:
total_acc = 0
for batch in test_loader_01:
    data,labels = batch
    predictions = adaboost_01(data).squeeze()
    batch_acc = np.sum(predictions == labels) / len(labels)
    total_acc += batch_acc
test_acc = total_acc / len(test_loader_01)
print(f"Test accuracy: {test_acc}")


Test accuracy: 1.0


## Test 

In [14]:
label_binary_tree.visualize()

  _[0, 1, 2]______      
 /                \     
[2]           _[0, 1]_  
             /        \ 
            [0]      [1]


In [13]:
train_loader,test_loader = get_dataloaders(n_qubits=n_qubits,n_layers=1,n_train_samples=n_train_samples,n_test_samples=n_test_samples,batch_size=batch_size,data_type=data_type)

In [20]:
predictions_012_list = []
predictions_01_list = []
labels_list = []

for batch in test_loader:
    data,labels = batch
    predictions_012_list.append(adaboost_012(data).squeeze())
    predictions_01_list.append(adaboost_01(data).squeeze())
    labels_list.append(labels)
    
predictions_012 = np.concatenate(predictions_012_list)
predictions_01 = np.concatenate(predictions_01_list)
labels = np.concatenate(labels_list)





In [25]:
# 将预测结果转换为最终分类
final_predictions = []
for pred_012, pred_01 in zip(predictions_012, predictions_01):
    if pred_012 == -1:
        # 如果predictions_012输出为-1，记录为类别0
        final_predictions.append(2)
    else:
        # 如果predictions_012输出为1，根据predictions_01决定
        if pred_01 == -1:
            final_predictions.append(0)
        else:
            final_predictions.append(1)

final_predictions = np.array(final_predictions)
final_predictions

array([0, 0, 1, ..., 1, 1, 0])

In [28]:
test_acc = np.sum(final_predictions == labels) / len(labels)
print(f"Test accuracy: {test_acc}")

Test accuracy: 1.0
